# Lucky Imaging – Procesamiento de Todos los Datasets
**Solarius – Nayely**

Este notebook aplica el método **Lucky Imaging + Strehl Proxy Ranking** a los 5 datasets del proyecto y genera:
- `Dataset Nayely.csv`
- `Dataset Jose.csv`
- `Dataset Benjamin.csv`
- `Dataset Extra.csv`
- `Dataset Joaquin.csv`
- `Promedios.txt`

### Método resumido
1. Normalizar cada frame del cubo FITS a [0, 1]
2. Calcular el **Strehl Proxy** (gradiente RMS) como medida de nitidez
3. Seleccionar el **top-20%** de frames más nítidos
4. **Shift-and-Add**: alinear por centroide y promediar
5. Comparar la **lucky image** contra el **promedio simple** (referencia)
6. Métricas: **MSE**, **PSNR**, **SSIM**

## 0. Configuración de rutas

Ajusta `DATA_ROOT` según dónde tengas los datasets FITS.

In [ ]:
import os  # necesario para el print de verificación

# ── Si usas Google Colab, monta Drive primero ──────────────
# from google.colab import drive
# drive.mount('/content/drive')

# ── Ruta raíz donde están las carpetas de datasets FITS ────
# Los datos están en Proyecto-Solarius/dataset/
# Este notebook está en Proyecto-Solarius/solarius_Nayely/
# → ruta relativa: ../dataset

DATA_ROOT = "../dataset"   # carpeta dataset/ en la raíz del proyecto

K_PERCENT   = 20    # top-K% de frames más nítidos
SAVE_IMAGES = True  # guardar lucky images como PNG
OUTPUT_DIR  = "."   # donde se guardan los CSVs y Promedios.txt (solarius_Nayely/)

# Datasets: nombre → (carpeta FITS, prefijo de archivos)
DATASETS = {
    "Nayely":   ("continuum_200p_0_99",       "continuum_200p_00"),
    "Jose":     ("chromosphere_100p_0_99",    "chromosphere_100p_00"),
    "Benjamin": ("chromosphere_100p_100_199", "chromosphere_100p_00"),
    "Extra":    ("chromosphere_200p_0_99",    "chromosphere_200p_00"),
    "Joaquin":  ("continuum_200p_100_199",    "continuum_200p_00"),
}

print(f"DATA_ROOT resuelto a: {os.path.abspath(DATA_ROOT)}")
print("Carpetas encontradas:", os.listdir(DATA_ROOT) if os.path.exists(DATA_ROOT) else "NO EXISTE")

## 1. Librerías

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy import ndimage
from skimage.metrics import mean_squared_error as mse_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
from PIL import Image

print("Librerías cargadas correctamente.")

## 2. Funciones Lucky Imaging

In [ ]:
def normalizacion_frame(frame):
    """Escala un frame 2D al rango [0, 1]."""
    mn, mx = frame.min(), frame.max()
    if mx == mn:
        return np.zeros_like(frame, dtype=np.float64)
    return (frame - mn) / (mx - mn)


def strehl_proxy(frame):
    """Proxy de nitidez basado en gradiente RMS (mayor = más nítido)."""
    gy = np.gradient(frame, axis=0)
    gx = np.gradient(frame, axis=1)
    return float(np.sqrt(gx**2 + gy**2).mean())


def select_top_k(frames, proxies, k_percent=20):
    """Selecciona el top-k% de frames ordenados de mejor a peor proxy."""
    n_keep  = max(1, int(len(proxies) * k_percent / 100))
    indices = np.argsort(proxies)[::-1][:n_keep]
    return frames[indices], indices


def centroide(frame):
    """Centro de masa de la imagen."""
    total = frame.sum() + 1e-10
    Y, X  = np.mgrid[:frame.shape[0], :frame.shape[1]]
    cy    = (Y * frame).sum() / total
    cx    = (X * frame).sum() / total
    return cy, cx


def shift_and_add(frames_to_stack):
    """Alinea frames al centroide del primero y los promedia (lucky image)."""
    ref_cy, ref_cx = centroide(frames_to_stack[0])
    aligned = []
    for frame in frames_to_stack:
        cy, cx = centroide(frame)
        dy = ref_cy - cy
        dx = ref_cx - cx
        aligned.append(ndimage.shift(frame, shift=(dy, dx), order=3))
    return np.clip(np.mean(aligned, axis=0), 0.0, 1.0)


def lucky_imaging(data, k_percent=K_PERCENT):
    """
    Aplica Lucky Imaging a un cubo de datos FITS.
    Retorna (lucky_image, simple_mean, indices_seleccionados).
    """
    frames      = np.array([normalizacion_frame(data[i]) for i in range(data.shape[0])])
    simple_mean = normalizacion_frame(frames.mean(axis=0))
    proxies     = np.array([strehl_proxy(f) for f in frames])
    selected, sel_idx = select_top_k(frames, proxies, k_percent)
    lucky_img   = shift_and_add(selected)
    return lucky_img, simple_mean, sel_idx


def compute_metrics(lucky_image, reference):
    """MSE, PSNR (dB) y SSIM entre lucky image y referencia."""
    m    = mse_fn(reference, lucky_image)
    p    = psnr_fn(reference, lucky_image, data_range=1.0)
    win  = min(7, reference.shape[0], reference.shape[1])
    win  = win if win % 2 != 0 else win - 1
    s    = ssim_fn(reference, lucky_image, data_range=1.0, win_size=win)
    return m, p, s


def save_png(arr, path):
    Image.fromarray((arr * 255).astype(np.uint8), mode="L").save(path)


print("Funciones Lucky Imaging definidas.")

## 3. Pipeline – Procesamiento de todos los datasets

In [ ]:
promedios = {}

for nombre, (carpeta_fits, prefijo) in DATASETS.items():
    print(f"\n{'='*55}")
    print(f"  Dataset: {nombre}  →  {carpeta_fits}")
    print(f"{'='*55}")

    dataset_dir = os.path.join(DATA_ROOT, carpeta_fits)
    archivos    = sorted(glob.glob(os.path.join(dataset_dir, f"{prefijo}*.fits")))

    if not archivos:
        print(f"  ⚠️  No se encontraron FITS en: {dataset_dir}")
        print(f"  Verifica DATA_ROOT o descarga el dataset desde zenodo.org/records/7064324")
        continue

    print(f"  {len(archivos)} archivos encontrados.")

    if SAVE_IMAGES:
        img_dir = os.path.join(OUTPUT_DIR, "output_png", f"lucky_{nombre.lower()}")
        os.makedirs(img_dir, exist_ok=True)

    resultados = []
    for fits_path in archivos:
        basename = os.path.basename(fits_path)
        try:
            with fits.open(fits_path) as hdul:
                data = hdul[0].data.astype(np.float64)

            lucky_img, ref, sel_idx = lucky_imaging(data)
            m, p, s = compute_metrics(lucky_img, ref)

            id_fila = f"{carpeta_fits}/{basename}"
            resultados.append({"ID": id_fila, "MSE": m, "PSNR_dB": p, "SSIM": s})

            if SAVE_IMAGES:
                save_png(lucky_img, os.path.join(img_dir, basename.replace(".fits", ".png")))

            print(f"  ✅ {basename} | frames={data.shape[0]} sel={len(sel_idx)} "
                  f"| MSE={m:.5f}  PSNR={p:.2f}dB  SSIM={s:.4f}")

        except Exception as e:
            print(f"  ❌ Error en {basename}: {e}")

    if resultados:
        df = pd.DataFrame(resultados)
        csv_path = os.path.join(OUTPUT_DIR, f"Dataset {nombre}.csv")
        df.to_csv(csv_path, index=False)
        promedios[nombre] = df[["MSE", "PSNR_dB", "SSIM"]].mean()
        print(f"\n  → CSV guardado: {csv_path}")

print("\n¡Procesamiento completo!")

## 4. Promedios por dataset → Promedios.txt

In [ ]:
promedios_path = os.path.join(OUTPUT_DIR, "Promedios.txt")
with open(promedios_path, "w", encoding="utf-8") as f:
    for nombre, avg in promedios.items():
        f.write(f"\n--- Dataset {nombre} Promedio de Métricas ---\n")
        f.write(avg.to_string() + "\n")

print("=" * 55)
print("  RESUMEN – Lucky Imaging")
print("=" * 55)
for nombre, avg in promedios.items():
    print(f"\n  Dataset {nombre}:")
    print(f"    MSE    = {avg['MSE']:.6f}")
    print(f"    PSNR   = {avg['PSNR_dB']:.4f} dB")
    print(f"    SSIM   = {avg['SSIM']:.6f}")

print(f"\n✅ Promedios guardados en: {promedios_path}")

## 5. Visualización de ejemplo (primer archivo del dataset Nayely)

In [ ]:
# Carga el primer archivo del dataset Nayely para visualizar el proceso
carpeta, prefijo = DATASETS["Nayely"]
ejemplo_path = sorted(glob.glob(os.path.join(DATA_ROOT, carpeta, f"{prefijo}*.fits")))

if ejemplo_path:
    with fits.open(ejemplo_path[0]) as hdul:
        data_ej = hdul[0].data.astype(np.float64)

    lucky_ej, mean_ej, sel_idx = lucky_imaging(data_ej)
    m_ej, p_ej, s_ej = compute_metrics(lucky_ej, mean_ej)

    frames_ej = np.array([normalizacion_frame(data_ej[i]) for i in range(data_ej.shape[0])])
    best_frame = frames_ej[sel_idx[0]]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    titles = [
        f"Mejor frame (proxy ranking)",
        f"Promedio simple ({data_ej.shape[0]} frames)",
        f"Lucky Image (top-{K_PERCENT}%: {len(sel_idx)} frames)\nMSE={m_ej:.5f}  PSNR={p_ej:.2f}dB  SSIM={s_ej:.4f}"
    ]
    imgs = [best_frame, mean_ej, lucky_ej]

    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(img, cmap="afmhot", vmin=0, vmax=1)
        ax.set_title(title, fontsize=9)
        ax.axis("off")

    plt.suptitle(f"Dataset Nayely – {os.path.basename(ejemplo_path[0])}", fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron archivos para visualizar. Verifica DATA_ROOT.")